# DFX Manager Register Readback Test

This notebook verifies that the `DFX_Mng` driver is consistent with the hardware by performing a
write/readback check on every R/W register in Bank 0 and all Bank 1 slot fields.

> **Board Required:** This notebook must run on a PYNQ board with the exported bitstream placed in `hw/`.
>
> **Destructive:** All writable registers are overwritten. Re-initialise the system after this test.

## Step 1 — Load Overlay

In [ ]:
import os
from pynq import Overlay
import driver.cap as cap

PRJ_DIR    = os.getcwd()
PRJ_HW_DIR = os.path.join(PRJ_DIR, 'hw')
FULL_BS    = 'system.bin'

cap.change_pl_config_mode('pcap', True, '')
overlay = Overlay(os.path.join(PRJ_HW_DIR, FULL_BS))
print('Overlay loaded.')

## Step 2 — Access DFX Manager

In [ ]:
dfx_ip = overlay.dfx_unified_0
dfx_mng = dfx_ip.dfx_mng

print(f'DFX_Mng offset : 0x{dfx_ip.DFX_MNG_OFFSET:05X}')
print(f'LIM_AMT_SLOT   : {dfx_mng.LIM_AMT_SLOT}')

## Step 3 — Print Current Status (Before Test)

In [ ]:
dfx_mng.print_debug()

## Step 4 — Run Register Readback Test

Tests every R/W register by writing a known pattern, reading back, and comparing.
Hardware-narrower registers are masked appropriately before comparison.

| Section | Registers tested |
|---|---|
| Bank 0 | `REG_LAST_SESSION`, `REG_AMT_QUERY`, `REG_AMT_QUERY_PER_ITER`, `REG_DMA_IP_ADDR`, `REG_PR_IP_ADDR`, `REG_INTR_ENA` |
| Bank 1 | All 12 slot fields for every slot (0 … LIM_AMT_SLOT-1) |

In [ ]:
result = dfx_mng.test_reg_readback()
print()
print('OVERALL:', 'PASS' if result else 'FAIL')

## Step 5 — Reset to Clean State

In [ ]:
dfx_mng.clear_engine()
print('Engine cleared.')